In [1]:
import os
import re
import SimpleITK as sitk
import numpy as np
import pandas as pd
from tqdm import tqdm
from radiomics import featureextractor
import logging

# 设置日志级别为ERROR，只显示错误信息
logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)
logger.setLevel(logging.ERROR)

# 禁用radiomics的详细日志
radiomics_logger = logging.getLogger('radiomics')
radiomics_logger.setLevel(logging.ERROR)

def extract_3d_features_original(imagePath, maskPath):
    """
    修改版：Sigma=[1.0, 3.0]，补充 T1.csv 中包含的所有滤镜和纹理特征
    """
    # 检查输入的图像路径和掩码路径是否为空
    if imagePath is None or maskPath is None:
        raise Exception('Error getting testcase!')

    # 检查文件是否存在
    if not os.path.exists(imagePath) or not os.path.exists(maskPath):
        raise Exception('Image or mask file does not exist!')

    # 1. 参数设置：Sigma 指定为 1.0 和 3.0
    settings = {
        'binWidth': 25,
        'sigma': [1.0, 3.0],  # 遵照您的指示，只保留1和3
        'Interpolator': sitk.sitkBSpline,
        'resampledPixelSpacing': [0.5, 0.5, 0.5],
        'voxelArrayShift': 0,
        'normalize': True,
        'normalizeScale': 100
    }

    # 创建Radiomics特征提取器
    extractor = featureextractor.RadiomicsFeatureExtractor(**settings)

    # 2. 启用 T1.csv 中包含的所有图像类型 (Filters)
    # 基础类型
    extractor.enableImageTypeByName('Original')
    extractor.enableImageTypeByName('Wavelet')
    
    # 补充 T1 有的滤镜
    extractor.enableImageTypeByName('LoG')         # 对应 T1 中的 log-sigma
    extractor.enableImageTypeByName('Square')      # 对应 T1 中的 square
    extractor.enableImageTypeByName('Squareroot')  # 对应 T1 中的 squareroot
    extractor.enableImageTypeByName('Logarithm')   # 对应 T1 中的 logarithm
    extractor.enableImageTypeByName('Exponential') # 对应 T1 中的 exponential
    extractor.enableImageTypeByName('Gradient')    # 对应 T1 中的 gradient

    # 3. 启用特征类别
    
    # 形状特征 (保留您代码中更全的3D形状列表)
    extractor.enableFeaturesByName(shape=[
        'VoxelVolume', 'MeshVolume', 'SurfaceArea', 'SurfaceVolumeRatio', 'Compactness1', 
        'Compactness2', 'Sphericity', 'SphericalDisproportion', 'Maximum3DDiameter', 
        'Maximum2DDiameterSlice', 'Maximum2DDiameterColumn', 'Maximum2DDiameterRow', 
        'MajorAxisLength', 'MinorAxisLength', 'LeastAxisLength', 'Elongation', 'Flatness'
    ])

    # 一阶特征 (保留您代码中的列表)
    extractor.enableFeaturesByName(firstorder=[
        'Energy', 'TotalEnergy', 'Entropy', 'Minimum', '10Percentile', '90Percentile', 
        'Maximum', 'Mean', 'Median', 'InterquartileRange', 'Range', 'MeanAbsoluteDeviation', 
        'RobustMeanAbsoluteDeviation', 'RootMeanSquared', 'StandardDeviation', 'Skewness', 
        'Kurtosis', 'Variance', 'Uniformity'
    ])

    # --- 补充：T1.csv 中包含的纹理特征 ---
    # T1.csv 包含 glcm, glrlm, glszm, gldm。
    # 这里直接启用整个类，确保不会漏掉具体指标。
    extractor.enableFeatureClassByName('glcm')
    extractor.enableFeatureClassByName('glrlm')
    extractor.enableFeatureClassByName('glszm')
    extractor.enableFeatureClassByName('gldm')
    
    # 注：T1.csv 中没有 NGTDM，如果您希望严格与 T1 保持一致，这里就不加 NGTDM。
    # 如果想保留 3D 组的优势，可以取消下面这行的注释：
    # extractor.enableFeatureClassByName('ngtdm') 

    # --------------------------------------------------------

    mask = sitk.ReadImage(maskPath)
    mask_array = sitk.GetArrayFromImage(mask)
    labels = np.unique(mask_array)
    
    labels = labels[labels != 0]  # 排除背景标签
    
    if len(labels) == 0:
        raise Exception('mask中没有找到有效标签')
    
    # 明确选择标签2（管壁）
    if 2 in labels:
        label = 2  # 管壁
    elif 1 in labels:
        label = 1  # 管腔
    else:
        label = labels[0]
    
    # 执行特征提取
    result = extractor.execute(imagePath, maskPath, label=label)

    # 提取特征名和特征值
    feature_name = []
    feature_cur = []
    for key, value in result.items():
        # 跳过诊断信息
        if key.startswith('diagnostics_'):
            continue
            
        try:
            float_value = float(value)
            feature_name.append(key)
            feature_cur.append(float_value)
        except (ValueError, TypeError):
            continue

    return feature_cur, feature_name

def process_3d_original_data():
    """
    直接处理原始3D数据，提取所有4个组的患者特征
    """
    # 根路径
    root_dir = "/home/tzl/project/复发分类数据/复发分类"
    
    # 定义所有要处理的组
    groups = [
        "复发/两端无狭窄",
        "复发/一端狭窄", 
        "非复发/两端无狭窄",
        "非复发/一端狭窄"
    ]
    
    # 匹配病人文件夹名中的编号和切片范围
    pattern = re.compile(r"(.+?)\s*(\d{6,12})\s*[（(](\d+)\s*-\s*(\d+)[）)]")
    
    all_features = []
    all_feature_names = set()
    patient_info = []
    
    total_patients = 0
    successful_patients = 0
    
    # 先统计总患者数用于进度条
    for group in groups:
        group_path = os.path.join(root_dir, group)
        if os.path.exists(group_path):
            total_patients += len([d for d in os.listdir(group_path) if os.path.isdir(os.path.join(group_path, d))])
    
    print(f"总共需要处理 {total_patients} 个患者")
    
    # 创建总体进度条
    with tqdm(total=total_patients, desc="总体进度") as pbar:
        for group in groups:
            group_path = os.path.join(root_dir, group)
            if not os.path.exists(group_path):
                print(f"警告: 组路径不存在: {group_path}")
                continue
            
            print(f"\n开始处理组: {group}")
            
            patient_folders = [d for d in os.listdir(group_path) if os.path.isdir(os.path.join(group_path, d))]
            
            for patient_folder in patient_folders:
                fixed_name = (
                    patient_folder.replace("　", " ")
                    .replace("（", "(")
                    .replace("）", ")")
                    .replace("–", "-")
                    .replace("—", "-")
                    .strip()
                )

                match = pattern.match(fixed_name)
                if not match:
                    print(f"未找到合法格式: {patient_folder}")
                    pbar.update(1)
                    continue

                name, pid, start, end = match.groups()
                start, end = int(start), int(end)
                case_id = f"{name.strip().replace(' ', '_')}_{pid}"

                case_path = os.path.join(group_path, patient_folder)

                # 找 mask 文件
                mask_file = None
                for f in os.listdir(case_path):
                    if f.endswith("_contour_slice_prediction.mhd") and "T1CE" not in f:
                        mask_file = os.path.join(case_path, f)
                        break
                if mask_file is None:
                    print(f"未找到mask文件: {case_path}")
                    pbar.update(1)
                    continue

                # 找 T1 文件
                t1_file = None
                for f in os.listdir(case_path):
                    if f.endswith("_img.mhd") and "T1CE" not in f:
                        t1_file = os.path.join(case_path, f)
                        break
                if t1_file is None:
                    print(f"未找到T1文件: {case_path}")
                    pbar.update(1)
                    continue

                try:
                    # 直接对原始3D文件提取特征
                    features, feature_names = extract_3d_features_original(t1_file, mask_file)
                    
                    if features is not None:
                        # 存储特征数据
                        feature_dict = {name: value for name, value in zip(feature_names, features)}
                        feature_dict['patient_id'] = case_id
                        feature_dict['group'] = group  # 添加组信息
                        feature_dict['start_slice'] = start
                        feature_dict['end_slice'] = end
                        feature_dict['total_slices'] = end - start + 1
                        feature_dict['label_used'] = 2  # 记录使用的标签
                        
                        all_features.append(feature_dict)
                        all_feature_names.update(feature_names)
                        patient_info.append({
                            'patient_id': case_id,
                            'group': group,
                            'original_folder': patient_folder,
                            'start_slice': start,
                            'end_slice': end,
                            'total_slices': end - start + 1,
                            'labels_found': str(np.unique(sitk.GetArrayFromImage(sitk.ReadImage(mask_file))))
                        })
                        
                        successful_patients += 1
                        pbar.set_postfix({
                            '成功': successful_patients,
                            '组': group.split('/')[-1]
                        })
                        
                except Exception as e:
                    print(f"处理患者失败 {case_id}: {e}")
                
                pbar.update(1)
    
    if not all_features:
        print("没有成功提取到任何特征")
        return
    
    # 创建对齐的特征表格
    all_feature_names = sorted(list(all_feature_names))
    aligned_features = []
    
    for feature_dict in all_features:
        aligned_row = [
            feature_dict['patient_id'], 
            feature_dict['group'],
            feature_dict['start_slice'],
            feature_dict['end_slice'],
            feature_dict['total_slices'],
            feature_dict.get('label_used', 2)
        ]
        for name in all_feature_names:
            aligned_row.append(feature_dict.get(name, np.nan))
        aligned_features.append(aligned_row)
    
    # 创建DataFrame
    column_names = ['patient_id', 'group', 'start_slice', 'end_slice', 'total_slices', 'label_used'] + all_feature_names
    feature_df = pd.DataFrame(aligned_features, columns=column_names)
    
    # 保存患者信息
    patient_info_df = pd.DataFrame(patient_info)
    
    # 保存结果
    output_excel = "/home/tzl/project/复发分类数据/radiomics_features_3D_all_groups.xlsx"
    with pd.ExcelWriter(output_excel) as writer:
        feature_df.to_excel(writer, sheet_name='影像组学特征', index=False)
        patient_info_df.to_excel(writer, sheet_name='患者信息', index=False)
    
    print(f"\n3D特征提取完成！")
    print(f"保存到: {output_excel}")
    print(f"总共处理了 {successful_patients}/{total_patients} 个患者")
    print(f"分组统计:")
    group_stats = feature_df['group'].value_counts()
    for group, count in group_stats.items():
        print(f"  {group}: {count} 个患者")
    
    # 显示提取的特征信息
    print(f"\n特征信息:")
    print(f"  总共提取了 {len(all_feature_names)} 个影像组学特征")
    print(f"  特征类型: 一阶统计特征 + 形状特征 + Wavelet变换特征")
    
    return feature_df

if __name__ == "__main__":
    process_3d_original_data()

总共需要处理 237 个患者


总体进度:   0%|          | 0/237 [00:00<?, ?it/s]


开始处理组: 复发/两端无狭窄


总体进度:  14%|█▍        | 34/237 [02:11<13:51,  4.10s/it, 成功=34, 组=两端无狭窄]


开始处理组: 复发/一端狭窄


...


开始处理组: 非复发/两端无狭窄


总体进度:  96%|█████████▌| 228/237 [12:38<00:31,  3.48s/it, 成功=228, 组=两端无狭窄]


开始处理组: 非复发/一端狭窄


总体进度: 100%|██████████| 237/237 [13:11<00:00,  3.34s/it, 成功=237, 组=一端狭窄]  



3D特征提取完成！
保存到: /home/tzl/project/复发分类数据/radiomics_features_3D_all_groups.xlsx
总共处理了 237/237 个患者
分组统计:
  非复发/两端无狭窄: 190 个患者
  复发/两端无狭窄: 34 个患者
  非复发/一端狭窄: 9 个患者
  复发/一端狭窄: 4 个患者

特征信息:
  总共提取了 1427 个影像组学特征
  特征类型: 一阶统计特征 + 形状特征 + Wavelet变换特征
